# Image Deepfake Detection using XceptionNet

This notebook reproduces the exact same methods from the Google Colab implementation.
It uses XceptionNet pre-trained on ImageNet to detect deepfake images from DFDC and Celeb-DF datasets.

In [ ]:
# Install tf-explain for Grad-CAM visualization
!pip install tf-explain

In [ ]:
# Import libraries
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.applications.xception import Xception, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.xception import decode_predictions
import requests
import cv2
import os
from pathlib import Path
from tqdm import tqdm

%matplotlib inline

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 1. Load Pre-trained Xception Model

In [ ]:
# Load pre-trained Xception model
model = tf.keras.applications.xception.Xception(weights='imagenet', include_top=True)
print("Model Summary:")
model.summary()

## 2. Fetch ImageNet Labels

In [ ]:
# Fetch labels from ImageNet
response = requests.get('https://storage.googleapis.com/download.tensorflow.org/data/imagenet_class_index.json')
imagenet_map = response.json()

# Convert to dictionary
imagenet_map = {v[1]: k for k, v in imagenet_map.items()}

print(f"Loaded {len(imagenet_map)} ImageNet labels")
print("Sample labels:", list(imagenet_map.items())[:5])

## 3. Image Preprocessing Function (Exactly as in Colab)

In [ ]:
def convert_for_xception(image_path):
    """
    Converts ANY image into Xception format:
    - Resize to 299x299
    - Convert grayscale → RGB
    - Add batch dimension
    - Normalize using preprocess_input
    """
    img = tf.keras.preprocessing.image.load_img(
        image_path,
        target_size=(299, 299)
    )
    
    img = tf.keras.preprocessing.image.img_to_array(img)
    
    # If grayscale, convert to 3 channels
    if img.shape[-1] == 1:
        img = np.repeat(img, 3, axis=-1)
    
    # Add batch dimension
    img = np.expand_dims(img, axis=0)
    
    # Preprocess for Xception
    img = tf.keras.applications.xception.preprocess_input(img)
    
    return img

print("Image conversion function defined.")

## 4. Test with Sample Image

In [ ]:
# Example: Test with a sample image path
# Replace with your actual image path
IMAGE_PATH = '/content/sample_data/1+NqBqX0PFttwjhzYB8X5A-w.webp'  # Example from Colab

# Check if file exists, if not create a test image
if not os.path.exists(IMAGE_PATH):
    print(f"Image not found at {IMAGE_PATH}")
    print("Please provide your image path.")
else:
    # Convert the image
    img_ready = convert_for_xception(IMAGE_PATH)
    
    # Make prediction
    pred = model.predict(img_ready)
    
    # Decode predictions (top-5)
    decoded_predictions = tf.keras.applications.xception.decode_predictions(pred, top=5)
    
    print("\nTop-5 Predictions:")
    for i, (imagenet_id, label, score) in enumerate(decoded_predictions[0]):
        print(f"{i+1}. {label}: {score:.4f}")

## 5. Load and Process Dataset (DFDC or Celeb-DF)

In [ ]:
# Configure dataset paths
DATASET_PATH = '../data/raw'  # Path to DFDC or Celeb-DF
REAL_IMAGES_PATH = os.path.join(DATASET_PATH, 'real')  # Real images
FAKE_IMAGES_PATH = os.path.join(DATASET_PATH, 'fake')  # Fake images

def load_images_from_directory(directory_path, label, max_images=None):
    """
    Load images from directory and preprocess them.
    
    Args:
        directory_path: Path to image directory
        label: Label (0 for real, 1 for fake)
        max_images: Maximum number of images to load
    
    Returns:
        Arrays of images and labels
    """
    images = []
    labels = []
    
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    
    if not os.path.exists(directory_path):
        print(f"Directory not found: {directory_path}")
        return np.array([]), np.array([])
    
    image_files = [
        f for f in os.listdir(directory_path)
        if os.path.splitext(f)[1].lower() in image_extensions
    ]
    
    if max_images:
        image_files = image_files[:max_images]
    
    for img_file in tqdm(image_files, desc=f"Loading from {directory_path}"):
        img_path = os.path.join(directory_path, img_file)
        try:
            # Load image
            img = cv2.imread(img_path)
            if img is None:
                continue
            
            # Convert BGR to RGB
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Resize to 299x299
            img = cv2.resize(img, (299, 299))
            
            # Normalize to [0, 1]
            img = img.astype('float32') / 255.0
            
            images.append(img)
            labels.append(label)
        except Exception as e:
            print(f"Error loading {img_file}: {e}")
            continue
    
    return np.array(images), np.array(labels)

print("Image loading function defined.")

In [ ]:
# Load dataset
print("\n=== Loading Dataset ===")

if os.path.exists(REAL_IMAGES_PATH) and os.path.exists(FAKE_IMAGES_PATH):
    print(f"Loading real images from {REAL_IMAGES_PATH}...")
    X_real, y_real = load_images_from_directory(REAL_IMAGES_PATH, label=0, max_images=100)
    
    print(f"Loading fake images from {FAKE_IMAGES_PATH}...")
    X_fake, y_fake = load_images_from_directory(FAKE_IMAGES_PATH, label=1, max_images=100)
    
    X = np.concatenate([X_real, X_fake], axis=0)
    y = np.concatenate([y_real, y_fake], axis=0)
    
    print(f"\nDataset loaded:")
    print(f"  Total images: {len(X)}")
    print(f"  Real images: {np.sum(y == 0)}")
    print(f"  Fake images: {np.sum(y == 1)}")
else:
    print(f"Dataset not found at {DATASET_PATH}")
    print(f"Expected structure:")
    print(f"  {REAL_IMAGES_PATH}/")
    print(f"  {FAKE_IMAGES_PATH}/")
    print("\nPlease download DFDC or Celeb-DF dataset and organize as above.")

## 6. Make Predictions on Dataset

In [ ]:
def predict_image_deepfake(image_path, model):
    """
    Predict if image is deepfake using Xception.
    
    Args:
        image_path: Path to image
        model: Xception model
    
    Returns:
        Prediction results
    """
    # Convert image for Xception
    img_ready = convert_for_xception(image_path)
    
    # Make prediction
    pred = model.predict(img_ready, verbose=0)
    
    # Decode predictions
    decoded = tf.keras.applications.xception.decode_predictions(pred, top=5)
    
    return {
        'predictions': decoded[0],
        'raw_output': pred[0]
    }

print("Prediction function defined.")

## 7. Visualize Predictions with Grad-CAM

In [ ]:
def visualize_image_with_predictions(image_path, model):
    """
    Load image and display with predictions.
    """
    # Load original image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Could not load image: {image_path}")
        return
    
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Get predictions
    result = predict_image_deepfake(image_path, model)
    
    # Display
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    ax.imshow(img)
    ax.axis('off')
    
    # Add predictions as title
    title = "Top-5 Predictions:\n"
    for i, (imagenet_id, label, score) in enumerate(result['predictions']):
        title += f"{i+1}. {label}: {score:.4f}\n"
    
    ax.set_title(title, fontsize=10, loc='left')
    plt.tight_layout()
    plt.show()

print("Visualization function defined.")

## 8. Batch Prediction on Dataset

In [ ]:
def batch_predict_images(image_directory, model, max_images=50):
    """
    Batch predict on multiple images.
    
    Args:
        image_directory: Directory containing images
        model: Xception model
        max_images: Maximum images to process
    
    Returns:
        List of predictions
    """
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    
    image_files = [
        f for f in os.listdir(image_directory)
        if os.path.splitext(f)[1].lower() in image_extensions
    ][:max_images]
    
    results = []
    
    for img_file in tqdm(image_files, desc="Predicting"):
        img_path = os.path.join(image_directory, img_file)
        try:
            result = predict_image_deepfake(img_path, model)
            results.append({
                'filename': img_file,
                'predictions': result['predictions']
            })
        except Exception as e:
            print(f"Error predicting {img_file}: {e}")
    
    return results

print("Batch prediction function defined.")

## 9. Example: Predict on Sample Images

In [ ]:
# Example: Predict on real images
if os.path.exists(REAL_IMAGES_PATH):
    print("\n=== Predicting on Real Images ===")
    real_results = batch_predict_images(REAL_IMAGES_PATH, model, max_images=5)
    
    for result in real_results:
        print(f"\n{result['filename']}:")
        for i, (imagenet_id, label, score) in enumerate(result['predictions'][:3]):
            print(f"  {i+1}. {label}: {score:.4f}")

# Example: Predict on fake images
if os.path.exists(FAKE_IMAGES_PATH):
    print("\n=== Predicting on Fake Images ===")
    fake_results = batch_predict_images(FAKE_IMAGES_PATH, model, max_images=5)
    
    for result in fake_results:
        print(f"\n{result['filename']}:")
        for i, (imagenet_id, label, score) in enumerate(result['predictions'][:3]):
            print(f"  {i+1}. {label}: {score:.4f}")

## 10. Save Model for Deployment

In [ ]:
# Save model for later use
MODEL_PATH = '../models/xception_deepfake.h5'
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

model.save(MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

## 11. Inference Function for Production

In [ ]:
def inference_image(image_path):
    """
    Production inference function.
    
    Args:
        image_path: Path to image
    
    Returns:
        Top-5 predictions with scores
    """
    # Convert image
    img_ready = convert_for_xception(image_path)
    
    # Predict
    pred = model.predict(img_ready, verbose=0)
    
    # Decode
    decoded = tf.keras.applications.xception.decode_predictions(pred, top=5)
    
    # Format results
    results = []
    for imagenet_id, label, score in decoded[0]:
        results.append({
            'label': label,
            'score': float(score),
            'imagenet_id': imagenet_id
        })
    
    return results

print("Inference function defined.")

## 12. Test Inference

In [ ]:
# Test inference on a sample image
test_image_path = '../data/raw/real/sample_image.jpg'  # Replace with actual image

if os.path.exists(test_image_path):
    print("\n=== Testing Inference ===")
    results = inference_image(test_image_path)
    
    print(f"\nResults for {test_image_path}:")
    for i, result in enumerate(results):
        print(f"{i+1}. {result['label']}: {result['score']:.4f}")
else:
    print(f"Test image not found at {test_image_path}")
    print("Please provide a valid image path.")